# Chapter 01: Day one

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "01"
__file__ = os.path.abspath("run.py")

## The scene

Inspector Varma does not look up when you come in. She slides a single page across the desk.
"Ivan Petrov. Night supervisor. He found the body." She points at the grey workstation in the
corner. "That is LANTERN. It runs here, offline, because evidence never leaves this building. Ask
it what Petrov's statement tells us. Then tell me what it got wrong."

You have one witness statement ([`case/statements/petrov.md`](../../case/statements/petrov.md)),
one local model, and no idea yet what a good question looks like.

## The concept

A language model does one thing: given some text, it predicts the next token (a word or a piece of
a word), appends it, and repeats. It has no goal of its own. Whatever you send is the only thing
it knows about what you want.

**Prompt engineering** is designing that input so the most likely continuation is the answer you
need. It is not magic phrasing. It is the same discipline as writing a good brief for a new
colleague: say what the task is, give the material, set the rules, and say what the result should
look like.

```mermaid
flowchart LR
    P["Your prompt<br/>(task + evidence + rules + format)"] --> T[Tokenizer<br/>text to token ids]
    T --> M[Model<br/>scores every possible next token]
    M --> S[Sampler<br/>picks one token]
    S --> O[Output so far]
    O -->|append and repeat| M
    S -->|stop token or limit| A[Answer]
```

Everything in this lab is a way of controlling one of those boxes: what goes into the prompt
(chapters 01 to 04, 07, 08, 11), how the sampler picks (05, 06), how much the model reasons before
answering (09, 10), which model sits in the middle (12), and how you prove the whole thing works
(13 to 15).

## How it works

A useful prompt has up to five parts. You will not always need all five, but you should know which
ones you left out.

![Anatomy of a prompt](../../docs/diagrams/01-prompt-anatomy.svg)

| Part | Question it answers | Petrov example |
|---|---|---|
| Task | What should the model do? | Build a timeline of 23:00 to 23:59. |
| Context | What material should it use? | The statement, inside `<statement>` tags. |
| Rules | What must it do or never do? | Quote word for word. Mark approximate times with `~`. |
| Format | What shape should the answer take? | `HH:MM \| event \| quote`, one per line. |
| Examples | What does a good answer look like? | (Not used here. Chapter 07 is about examples.) |

The vague prompt in this chapter has only the context. The precise prompt has the first four.

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import re

from lantern import case, chat, learn
from lantern.ui import chapter, evidence, finding, note, reply, save_shot, step, your_turn

chapter("01", "Day one", """
Inspector Varma does not look up when you come in. She slides a single page across the desk.
"Ivan Petrov. Night supervisor. He found the body." Then she points at the grey workstation
in the corner. "That is LANTERN. It runs here, offline, because evidence never leaves this
building. Ask it what Petrov's statement tells us. Then tell me what it got wrong."
""")

statement = case.read("statements/petrov.md")
evidence("case/statements/petrov.md", statement)
note("This statement is the only material. Both prompts below send it to the same model.")

### 1. The vague prompt

In [ ]:
step("1. The vague prompt")
vague = f"Summarise this.\n\n{statement}"
note("Prompt: 'Summarise this.' plus the statement. No task, no focus, no format.")
note("Watch for: does the reply notice anything odd about the times? Does it add facts?")
p_gap = learn.predict(
    "Will the reply to 'Summarise this.' work out how long Petrov took between the bang and "
    "finding the body?",
    ["Yes, it states the gap in minutes", "No, it summarises without the gap"],
)
r = chat(vague)
reply("LANTERN, vague prompt", r.text, meta=f"{r.prompt_tokens} tokens in, {r.output_tokens} out")
STATED = {"18:00", "21:00", "23:00", "23:30", "23:52", "02:10"}  # every time the statement gives
GAP = r"\b(1[0-3]|ten|eleven|twelve|thirteen)[ -]?min"  # the 12 minutes, however it is written


def times_in(text: str) -> set[str]:
    """Every HH:MM in a reply, without a leading ~."""
    return {f"{int(h):02d}:{m}" for h, m in re.findall(r"\b(\d{1,2}):(\d{2})\b", text)}


vague_gap = bool(re.search(GAP, r.text, re.I))
learn.reveal(
    p_gap,
    answer=0 if vague_gap else 1,
    why="'Summarise' names no target, so the reply is whatever the model thinks matters. The gap "
    "only appears if something in the prompt asks about time taken.",
)

### 2. The precise prompt

In [ ]:
step("2. The precise prompt")
precise = f"""You are reviewing a witness statement for a police inquiry.

Task: build a timeline of what the witness says happened between 23:00 and 23:59.

Rules:
- One line per event, in time order, formatted as `HH:MM | event | quote`.
- The quote must be copied word for word from the statement.
- Convert spoken times to 24-hour time. The night shift runs 18:00 to 06:00, so "twenty to
  twelve" means 23:40.
- If the witness gives an approximate time, write it as `~HH:MM`.
- After the timeline, list any gap where the witness's account does not explain the time taken.
  Use the walking distances the witness gives.

Statement:
<statement>
{statement}
</statement>"""
note("Same statement. Now the prompt says what to do, how to format it, and what to look for.")
note("It is the `precise` variable in run.py: task, rules, format, then the statement in tags.")
p_tokens = learn.predict_number(
    f"The vague prompt cost {r.prompt_tokens} tokens in. How many will the precise prompt cost?",
    unit="tokens",
)
p_2340 = learn.predict(
    "Petrov says the bang came 'at about twenty to twelve'. What time will the timeline give it?",
    ["23:40 or ~23:40", "Some other time", "No time for the bang at all"],
)
p_quotes = learn.predict(
    "The rules say every quote is copied word for word. How many of the timeline's quotes will "
    "really be in the statement?",
    ["Every one", "Most, not all", "Half or fewer", "There will be no timeline lines"],
)
r2 = chat(precise, temperature=0)
reply("LANTERN, precise prompt", r2.text,
      meta=f"{r2.prompt_tokens} tokens in, {r2.output_tokens} out")
learn.reveal(
    p_tokens,
    actual=r2.prompt_tokens,
    tolerance=0.3,
    why=f"Task, rules and tags added {r2.prompt_tokens - r.prompt_tokens} tokens to the same "
    "statement. Structure is cheap next to the evidence it frames.",
)
flat = " ".join(statement.split())
rows = [ln for ln in r2.text.splitlines() if re.match(r"\s*[-*]?\s*~?\d{1,2}:\d{2}\s*\|", ln)]
quotes = [ln.split("|", 2)[2].strip().strip("`\"' ") for ln in rows if ln.count("|") >= 2]
verbatim = sum(1 for q in quotes if q and " ".join(q.split()) in flat)
bang = next((ln for ln in rows if re.search(r"bang", ln, re.I)), "")
bang_at = 0 if re.search(r"~?23:40", bang) else (1 if bang else 2)
learn.reveal(
    p_2340,
    answer=bang_at,
    why="'Twenty to twelve' is 23:40 only because the rules say the shift runs at night. A model "
    "cannot know the shift pattern unless the prompt tells it.",
)
learn.reveal(
    p_quotes,
    answer=3 if not rows else (0 if verbatim == len(rows) else (1 if verbatim > len(rows) / 2
                                                                else 2)),
    why=f"{verbatim} of {len(rows)} timeline quotes match the statement word for word. Asking for "
    "quotes makes the reply checkable; it does not make it true. Check them.",
)

### 3. What changed

In [ ]:
step("3. What changed")
note("The model and the evidence were identical. Only the prompt changed.")
note("The vague prompt gave a summary. The precise prompt gave an artefact you can check.")
note("Check the quotes against the statement yourself. LANTERN is a witness too.")

your_turn(
    "01",
    check=[
        "Petrov hears a bang, then later finds the body. Which reply worked out how long that "
        "took, and whether the walk he describes explains it?",
        "Open case/statements/petrov.md and compare every quote in the precise reply with it. "
        "Is any quote, time or distance not in the statement? That is the model inventing.",
        "Did 'twenty to twelve' become 23:40? Only the rule about the night shift makes it so.",
    ],
    exercises=[
        "In run.py, delete the <statement> and </statement> tags in the `precise` prompt. "
        "Does the timeline format still hold?",
        "In run.py, change 'copied word for word' to 'summarised'. How many details survive?",
        "In run.py, add to the Task line: 'Calculate how long the witness took to reach "
        "stairwell C after the bang.' Does the model find the gap on its own?",
        "No edit: run LANTERN_MODEL=qwen3:4b make lesson N=01 and compare the bigger model.",
    ],
)

learn.record(
    vague_prompt_tokens=r.prompt_tokens,
    precise_prompt_tokens=r2.prompt_tokens,
    vague_found_gap=vague_gap,
    precise_found_gap=bool(re.search(GAP, r2.text, re.I)),
    vague_invented_times=len(times_in(r.text) - STATED - {"23:40"}),
    precise_invented_times=len(times_in(r2.text) - STATED - {"23:40"}),
    timeline_lines=len(rows),
    quotes_verbatim=verbatim,
    bang_at_2340=bang_at == 0,
)

finding("""
Petrov says he heard a bang at about 23:40. He says the hut is two minutes from stairwell C.
He says he found Reyes at 23:52. That is about twelve minutes to cover a two minute walk, and
he explains it with a cup of tea and a detour past the reefer stacks.

Varma reads the timeline twice. "Good. Now find out why he waited."
""")
save_shot("01-prompt-engineering")

## The nuances

**1. "Summarise" is a task with no target.** A summary is whatever the model thinks matters, and a
small model thinks the first and last paragraphs matter. A timeline with quotes is a task with a
checkable output. Prefer tasks whose answers you can verify.

**2. Delimiters stop evidence from being read as instructions.** Wrapping the statement in
`<statement>` tags tells the model where your instructions end and the material begins. Without
them, a document that says "ignore the above" is just more instructions. This matters more when the
material comes from a suspect.

```mermaid
flowchart TB
    subgraph Without delimiters
      A1["Summarise this. I went past the reefer stacks first..."] --> B1[One stream of text:<br/>where does the task end?]
    end
    subgraph With delimiters
      A2["Task + rules"] --> C2[Model]
      D2["&lt;statement&gt; ... &lt;/statement&gt;"] --> C2
    end
```

**3. Ask for quotes, then check them.** A model that must quote is easier to audit than a model that
paraphrases. Small models still invent quotes, timings and distances. In the run below, the 1.7B
model may produce walking times that are not in the statement. That is the point of Varma's
instruction: *tell me what it got wrong.* LANTERN is a witness too.

**4. Small models need more structure, not less.** A frontier model may infer that you want a
timeline from "what does this tell us?". A 1.7B model on a laptop will not. Everything you make
explicit is something the model does not have to guess.

**5. The prompt is code.** It has inputs, rules and an output contract. Later chapters version it
(13), test it (14) and read its failures (15).

## On a hosted model

```bash
make lesson N=01 TIER=hosted     # needs ANTHROPIC_API_KEY in .env and `uv sync --extra cloud`
make lesson N=01 TIER=hosted REPLAY=1   # once a run has been recorded, no key needed
```

The hosted path sends both prompts to claude-haiku-5-5 (the `default` alias in `config.yaml`) as
plain user turns: this chapter uses no system prompt, no schema and no thinking, and with
`think: false` in the defaults `cloud.py` sends thinking disabled. Claude does not take
`temperature`, `top_k`, `top_p` or `seed`, so the precise prompt's `temperature=0` is left out and
reported on the console (see `src/lantern/cloud.py`). No step is skipped on this tier.

| Measure | Local: qwen3:1.7b, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| Time the vague reply invented | "approximately 22:52" | not run |
| Precise reply states the 23:40 to 23:52 gap | no: it lists the times, the sum is in code | not run |
| "Twenty to twelve" read as, with the rule | 23:40 | not run |
| Precise timeline includes the bang at 23:40 | no | not run |

Local numbers: qwen3:1.7b, Ollama, Apple M1 Pro 16 GB, run of 23 November 2026. Live hosted
numbers are not run: this lab has no hosted key.

What the hosted run will test: whether the invented time and the dropped 23:40 are small-model
effects, and whether a larger model does the 12-minute sum itself. That the vague prompt gives an answer you
cannot check, and that the shift rule is needed for "twenty to twelve", should hold on both.

## Try this

1. Remove the `<statement>` tags and rerun. Does the format still hold?
2. Change the rule "copied word for word" to "summarised". How many details survive?
3. Add one line to the task: "Calculate how long the witness took to reach stairwell C after the
   bang." Does the model find the gap on its own now?
4. Run with the bigger model: `LANTERN_MODEL=qwen3:4b make lesson N=01`. Compare.

## Master it

After this chapter you can name the parts of a precise prompt, build one whose evidence cannot
pass itself off as instructions, and check a reply against its source.

**Predict, then see.** `make lesson N=01` stops four times to ask what will happen: whether
"Summarise this." finds the gap, how many tokens the precise prompt costs, what time the bang
gets in the timeline, and how many of its quotes are really in the statement. Each answer is
checked against your own run, not a fixed key. Add `ARGS=--no-predict` to watch without
stopping.

**Build it.** Write the prompt builder and its check in
[`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=01          # 8 offline tests, no model needed
make hint N=01              # one hint at a time
make solution N=01          # the reference, once you have tried
```

**Check yourself.** `make quiz N=01`: six questions, each with the reason behind the answer.

**Break it.** `make challenge N=01` checks your latest run against three targets: the shift rule
taken away, so the bang gets the wrong time; the vague prompt rescued with one sentence; and a
timeline in which every quote checks.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 01

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 01

In [ ]:
%run ../../scripts/learn.py hint 01

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 01

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

Petrov says he heard a bang at about 23:40. He says the hut is two minutes from stairwell C. He
says he found Reyes at 23:52. That is about twelve minutes to cover a two minute walk, and he
explains it with a cup of tea and a detour past the reefer stacks.

Varma reads the timeline twice. "Good. Now find out why he waited."

**Next:** [Chapter 02: Standing orders](../02-system-prompt/README.md)